# RetailHero

RetailHero is a real-world retail dataset released by **X5 Retail Group** for the *X5 Retail Hero: Uplift Modeling for Promotional Campaign* competition held in 2019–2020.

The task is to estimate how a promotional communication changes a customer's likelihood of making a purchase. Instead of predicting who is most likely to buy, uplift modeling focuses on customers whose behavior is most likely to change **because of the communication**.

The dataset is split across several tables:

- `clients`: customer information.
- `products`: product metadata.
- `purchases`: historical transactions before the campaign.
- `uplift_train`: treatment assignment and observed outcome.
- `uplift_test`: unlabeled customers used in the original competition.
- `uplift_sample_submission`: submission template for the competition.

## Relationship Between Tables

The main tables are connected through `client_id` and `product_id`:

```text

clients
   │
   │ client_id
   ├──────────────────────┐
   │                      │
   ▼                      ▼
purchases            uplift_train
   │                      │
   │ product_id           │
   ▼                      │
products                  │
                          │
                treatment_flg + target

```

## Why RetailHero Fits This Framework

RetailHero is useful for testing whether the uplift framework can work beyond a ready-made modeling table.

Unlike Hillstrom, customer features are not provided directly. They must first be built from customer information, product metadata, and purchase history. After this dataset-specific preparation, RetailHero can be converted into the same input expected by the framework:

```text
feature columns
treatment
outcome
```

The framework can then run the same training, validation, model selection, and locked-test workflow without RetailHero-specific modeling logic.

This makes RetailHero a useful test of whether the framework depends on a **shared prepared-data contract** rather than the original structure of a particular dataset.

In [228]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

COLORS = {
    "pacific_green": "#017360",
    "rainy_lake": "#446E8F",
    "serene_sea": "#77A7C4",
    "sora_blue": "#A1DBF1",
    "afterglow": "#F2E6CE",
    "bungalow_maple": "#F3D094",
}

PALETTE = list(COLORS.values())

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.labelsize":11,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [229]:
PROJECT_ROOT = Path.cwd().parents[1]

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "retailhero"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim" / "retailhero"

RAW_FILES = {
    "clients": RAW_DATA_DIR / "clients.csv",
    "products": RAW_DATA_DIR / "products.csv",
    "purchases": RAW_DATA_DIR / "purchases.csv",
    "uplift_train": RAW_DATA_DIR / "uplift_train.csv",
}

INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

for name, path in RAW_FILES.items():
    assert path.exists(), f"Missing {name}: {path}"

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_DIR)
print("Interim data:", INTERIM_DATA_DIR)

Project root: d:\thao\d\uplif_model\uplif_customer_selection
Raw data: d:\thao\d\uplif_model\uplif_customer_selection\data\raw\retailhero
Interim data: d:\thao\d\uplif_model\uplif_customer_selection\data\interim\retailhero


In [230]:
file_inventory = pd.DataFrame([
    {
        "table": table_name,
        "file": path.name,
        "size_mb": path.stat().st_size / (1024**2),
        "path": str(path),
    }
    for table_name, path in RAW_FILES.items()
]).sort_values("size_mb", ascending=False)

display(file_inventory)

,table,file,size_mb,path
2,purchases,purchases.csv,"4,256.9881",d:\thao\d\uplif_model\uplif_customer_selection...
0,clients,clients.csv,20.7293,d:\thao\d\uplif_model\uplif_customer_selection...
1,products,products.csv,3.7101,d:\thao\d\uplif_model\uplif_customer_selection...
3,uplift_train,uplift_train.csv,2.8616,d:\thao\d\uplif_model\uplif_customer_selection...


## DuckDB Loading Strategy

The raw RetailHero files are intentionally **not loaded into pandas as full tables**.

The loading strategy is:

1. Keep the original CSV files immutable under `data/raw/retailhero/`.
2. Use DuckDB as the query engine for every raw table.
3. Convert only the very large `purchases.csv` file to a local Parquet cache once.
4. Query `purchases` through DuckDB from Parquet on subsequent notebook runs.
5. Never run `SELECT * FROM purchases` into pandas.
6. Only convert small query results, summaries, samples, or final customer-level feature tables to pandas.

Why cache `purchases`?

- CSV is row-oriented and must be parsed repeatedly.
- Parquet is columnar and compressed.
- DuckDB can push column selection and filters into Parquet scans.
- The raw CSV remains the source of truth; the Parquet file is only an analytical cache.

The other RetailHero tables are much smaller, so keeping them as lazy DuckDB views over the original CSV files is sufficient.

In [231]:
DUCKDB_CACHE_DIR = INTERIM_DATA_DIR / "_duckdb_cache"
DUCKDB_TEMP_DIR = DUCKDB_CACHE_DIR / "tmp"

DUCKDB_CACHE_DIR.mkdir(parents=True,exist_ok=True,)
DUCKDB_TEMP_DIR.mkdir(parents=True,exist_ok=True,)

duckdb_connection = duckdb.connect(database=":memory:")


def to_sql_path(path: Path) -> str:
    """Return an absolute path escaped for use inside a SQL string literal."""
    return (
        path.resolve()
        .as_posix()
        .replace("'", "''")
    )


duckdb_connection.execute(
    f"SET temp_directory = '{to_sql_path(DUCKDB_TEMP_DIR)}'"
)

print("DuckDB temp directory:",DUCKDB_TEMP_DIR,)


DuckDB temp directory: d:\thao\d\uplif_model\uplif_customer_selection\data\interim\retailhero\_duckdb_cache\tmp


In [232]:
PURCHASES_PARQUET_PATH = (DUCKDB_CACHE_DIR / "purchases.parquet")

if not PURCHASES_PARQUET_PATH.exists():
    print("Creating purchases Parquet cache...")

    duckdb_connection.execute(
        f"""
        COPY (
            SELECT *
            FROM read_csv(
                '{to_sql_path(RAW_FILES["purchases"])}',
                header = true
            )
        )
        TO '{to_sql_path(PURCHASES_PARQUET_PATH)}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )
else:
    print("Using existing purchases Parquet cache.")

Using existing purchases Parquet cache.


In [233]:
CSV_VIEW_TABLES = [
    "clients",
    "products",
    "uplift_train",
]

for table_name in CSV_VIEW_TABLES:
    file_path = RAW_FILES[table_name]

    duckdb_connection.execute(
        f"""
        CREATE OR REPLACE VIEW {table_name}_raw AS
        SELECT *
        FROM read_csv(
            '{to_sql_path(file_path)}',
            header = true,
            sample_size = 100000,
            strict_mode = true
        )
        """
    )

# The large purchases table is queried from the Parquet cache.
duckdb_connection.execute(
    f"""
    CREATE OR REPLACE VIEW purchases_raw AS
    SELECT *
    FROM read_parquet(
        '{to_sql_path(PURCHASES_PARQUET_PATH)}'
    )
    """
)

registered_relations = duckdb_connection.execute(
    "SHOW TABLES"
).df()

display(registered_relations)

,name
0,clients_raw
1,products_raw
2,purchases_raw
3,uplift_train_raw


## Data Overview

## Table and Column Description

The RetailHero dataset contains customer information, product metadata, historical purchase transactions, and uplift experiment labels.

### `clients`

One row represents one customer.

| Column              | Description                                            |
| ------------------- | ------------------------------------------------------ |
| `client_id`         | Unique customer identifier used to link tables         |
| `first_issue_date`  | Date when the customer's loyalty card was first issued |
| `first_redeem_date` | Date when loyalty points were first redeemed           |
| `age`               | Customer age                                           |
| `gender`            | Customer gender                                        |

### `products`

One row represents one product.

| Column             | Description                                              |
| ------------------ | -------------------------------------------------------- |
| `product_id`       | Unique product identifier used to link purchase records  |
| `level_1`          | Highest-level product category                           |
| `level_2`          | Second-level product category                            |
| `level_3`          | Third-level product category                             |
| `level_4`          | Most detailed product category provided                  |
| `segment_id`       | Product segment identifier                               |
| `brand_id`         | Brand identifier                                         |
| `vendor_id`        | Vendor or supplier identifier                            |
| `netto`            | Product net weight                                       |
| `is_own_trademark` | Whether the product belongs to the retailer's own brand  |
| `is_alcohol`       | Whether the product is an alcoholic product              |

### `purchases`

One row represents a product line within a transaction.  
A single transaction can therefore appear on multiple rows when multiple products were purchased.

| Column                    | Description                                         |
| ------------------------- | --------------------------------------------------- |
| `client_id`               | Customer who made the purchase                      |
| `transaction_id`          | Transaction identifier                              |
| `transaction_datetime`    | Date and time of the transaction                    |
| `regular_points_received` | Regular loyalty points earned                       |
| `express_points_received` | Express loyalty points earned                       |
| `regular_points_spent`    | Regular loyalty points redeemed                     |
| `express_points_spent`    | Express loyalty points redeemed                     |
| `purchase_sum`            | Purchase amount recorded for the product line       |
| `store_id`                | Store identifier                                    |
| `product_id`              | Purchased product identifier                        |
| `product_quantity`        | Quantity of the product purchased                   |
| `trn_sum_from_iss`        | Transaction amount associated with point issuance   |
| `trn_sum_from_red`        | Transaction amount associated with point redemption |

### `uplift_train`

One row represents one customer in the labeled uplift experiment.

| Column          | Description                                                                                                |
| --------------- | ---------------------------------------------------------------------------------------------------------- |
| `client_id`     | Customer identifier used to join with customer and purchase data                                           |
| `treatment_flg` | Treatment assignment: `1` if the customer received the promotional communication, otherwise `0` for control |
| `target`        | Binary outcome indicating whether the customer made a purchase after treatment assignment                  |

For the prepared uplift dataset:

```text
treatment = treatment_flg
outcome   = target

In [234]:
RELATIONS = {
    "clients": "clients_raw",
    "products": "products_raw",
    "purchases": "purchases_raw",
    "uplift_train": "uplift_train_raw",
}

table_shapes = []

for table_name, relation_name in RELATIONS.items():
    row_count = duckdb_connection.execute(
        f"SELECT COUNT(*) FROM {relation_name}"
    ).fetchone()[0]

    schema = duckdb_connection.execute(
        f"DESCRIBE {relation_name}"
    ).df()

    table_shapes.append({
        "table": table_name,
        "rows": row_count,
        "columns": len(schema),
    })

table_shapes = pd.DataFrame(table_shapes)

display(table_shapes)

,table,rows,columns
0,clients,400162,5
1,products,43038,11
2,purchases,45786568,13
3,uplift_train,200039,3


In [235]:
for table_name, relation_name in RELATIONS.items():
    print(f"\n{table_name}")
    display(
        duckdb_connection.execute(
            f"DESCRIBE {relation_name}"
        ).df()
    )


clients


,column_name,column_type,null,key,default,extra
0,client_id,VARCHAR,YES,None,None,None
1,first_issue_date,TIMESTAMP,YES,None,None,None
2,first_redeem_date,TIMESTAMP,YES,None,None,None
3,age,BIGINT,YES,None,None,None
4,gender,VARCHAR,YES,None,None,None



products


,column_name,column_type,null,key,default,extra
0,product_id,VARCHAR,YES,None,None,None
1,level_1,VARCHAR,YES,None,None,None
2,level_2,VARCHAR,YES,None,None,None
3,level_3,VARCHAR,YES,None,None,None
4,level_4,VARCHAR,YES,None,None,None
5,segment_id,DOUBLE,YES,None,None,None
6,brand_id,VARCHAR,YES,None,None,None
7,vendor_id,VARCHAR,YES,None,None,None
8,netto,DOUBLE,YES,None,None,None
9,is_own_trademark,BIGINT,YES,None,None,None



purchases


,column_name,column_type,null,key,default,extra
0,client_id,VARCHAR,YES,None,None,None
1,transaction_id,VARCHAR,YES,None,None,None
2,transaction_datetime,TIMESTAMP,YES,None,None,None
3,regular_points_received,DOUBLE,YES,None,None,None
4,express_points_received,DOUBLE,YES,None,None,None
5,regular_points_spent,DOUBLE,YES,None,None,None
6,express_points_spent,DOUBLE,YES,None,None,None
7,purchase_sum,DOUBLE,YES,None,None,None
8,store_id,VARCHAR,YES,None,None,None
9,product_id,VARCHAR,YES,None,None,None



uplift_train


,column_name,column_type,null,key,default,extra
0,client_id,VARCHAR,YES,None,None,None
1,treatment_flg,BIGINT,YES,None,None,None
2,target,BIGINT,YES,None,None,None


In [236]:
REQUIRED_COLUMNS = {
    "clients": {"client_id"},
    "products": {"product_id"},
    "purchases": {
        "client_id",
        "transaction_id",
        "transaction_datetime",
        "product_id",
    },
    "uplift_train": {
        "client_id",
        "treatment_flg",
        "target",
    },
}

for table_name, relation_name in RELATIONS.items():
    columns = {
        row[0]
        for row in duckdb_connection.execute(
            f"DESCRIBE {relation_name}"
        ).fetchall()
    }

    missing = REQUIRED_COLUMNS[table_name] - columns

    if missing:
        raise ValueError(
            f"{table_name}: missing required columns {sorted(missing)}"
        )

print("Required RetailHero columns are present.")

Required RetailHero columns are present.


In [237]:
for table_name in (
    "clients",
    "products",
    "purchases",
    "uplift_train",
):
    print(f"\n{table_name}")

    display(
        duckdb_connection.execute(
            f"SELECT * FROM {RELATIONS[table_name]} LIMIT 5"
        ).df()
    )


clients


,client_id,first_issue_date,first_redeem_date,age,gender
0,000012768d,2017-08-05 15:40:48,2018-01-04 19:30:07,45,U
1,000036f903,2017-04-10 13:54:23,2017-04-23 12:37:56,72,F
2,000048b7a6,2018-12-15 13:33:11,NaT,68,F
3,000073194a,2017-05-23 12:56:14,2017-11-24 11:18:01,60,F
4,00007c7133,2017-05-22 16:17:08,2018-12-31 17:17:33,67,U



products


,product_id,level_1,level_2,level_3,level_4,segment_id,brand_id,vendor_id,netto,is_own_trademark,is_alcohol
0,0003020d3c,c3d3a8e8c6,c2a3ea8d5e,b7cda0ec0c,6376f2a852,123.0000,394a54a7c1,9eaff48661,0.4000,0,0
1,0003870676,e344ab2e71,52f13dac0c,d3cfe81323,6dc544533f,105.0000,acd3dd483f,10486c3cf0,0.6800,0,0
2,0003ceaf69,c3d3a8e8c6,f2333c90fb,419bc5b424,f6148afbc0,271.0000,f597581079,764e660dda,0.5000,0,0
3,000701e093,ec62ce61e3,4202626fcb,88a515c084,48cf3d488f,172.0000,54a90fe769,03c2d70bad,0.1120,0,0
4,0007149564,e344ab2e71,52f13dac0c,d3cfe81323,6dc544533f,105.0000,63417fe1f3,f329130198,0.6000,0,0



purchases


,client_id,transaction_id,transaction_datetime,regular_points_received,express_points_received,regular_points_spent,express_points_spent,purchase_sum,store_id,product_id,product_quantity,trn_sum_from_iss,trn_sum_from_red
0,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0000,0.0000,0.0000,0.0000,"1,007.0000",54a4a11a29,9a80204f78,2.0000,80.0000,NaN
1,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0000,0.0000,0.0000,0.0000,"1,007.0000",54a4a11a29,da89ebd374,1.0000,65.0000,NaN
2,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0000,0.0000,0.0000,0.0000,"1,007.0000",54a4a11a29,0a95e1151d,1.0000,24.0000,NaN
3,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0000,0.0000,0.0000,0.0000,"1,007.0000",54a4a11a29,4055b15e4a,2.0000,50.0000,NaN
4,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0000,0.0000,0.0000,0.0000,"1,007.0000",54a4a11a29,a685f1916b,1.0000,22.0000,NaN



uplift_train


,client_id,treatment_flg,target
0,000012768d,0,1
1,000036f903,1,1
2,00010925a5,1,1
3,0001f552b0,1,1
4,00020e7b18,1,1


## Structural Data Quality Checks

Before EDA or feature engineering, validate both dataset structure and value quality:

1. Primary keys are unique and non-null where required;
2. Treatment and target contain valid binary values;
3. Relationships between tables do not contain orphan records;
4. Numeric columns have plausible distributions and ranges;
5. Missing values are understood for every column;
6. Categorical values are internally consistent;
7. Explicit domain constraints are satisfied;
8. Datetime values and date relationships are plausible.


In [238]:
validation_issues = []

UNIQUE_KEYS = {
    "clients": "client_id",
    "products": "product_id",
    "uplift_train": "client_id",
}

key_profiles = []

for table_name, key in UNIQUE_KEYS.items():
    rows, null_keys, distinct_keys = duckdb_connection.execute(
        f""" 
        SELECT
            COUNT(*),
            COUNT(*) FILTER (WHERE {key} IS NULL),
            COUNT(DISTINCT {key})
        FROM {RELATIONS[table_name]}
        """
    ).fetchone()

    duplicate_keys = rows - null_keys - distinct_keys

    key_profiles.append({
        "table": table_name,
        "rows": rows,
        "distinct_keys": distinct_keys,
        "null_keys": null_keys,
        "duplicate_keys": duplicate_keys,
    })

    if null_keys or duplicate_keys:
        validation_issues.append(
            f"{table_name}.{key}: {null_keys} null keys, {duplicate_keys} duplicate keys"
    )

display(pd.DataFrame(key_profiles))

,table,rows,distinct_keys,null_keys,duplicate_keys
0,clients,400162,400162,0,0
1,products,43038,43038,0,0
2,uplift_train,200039,200039,0,0


In [239]:
treatment_target_check = duckdb_connection.execute(
    """
    SELECT 
        COUNT(*) FILTER (WHERE treatment_flg IS NULL) AS treatment_nulls,
        COUNT(*) FILTER(
            WHERE treatment_flg IS NOT NULL
                AND treatment_flg NOT IN (0,1)
        ) AS treatment_invalid,
        COUNT(*) FILTER (WHERE target IS NULL) AS target_nulls,
        COUNT(*) FILTER (
            WHERE target IS NOT NULL
                AND target NOT IN (0,1)
        ) AS target_invalid
    FROM uplift_train_raw
    """
).df()

display(treatment_target_check)

if treatment_target_check.iloc[0].sum() > 0:
    validation_issues.append("uplift_train: invalid or missing treatment/target values")

treatment_target_profile = duckdb_connection.execute(
    """
    SELECT
        treatment_flg,
        target,
        COUNT(*) AS customers,
        COUNT(*) * 1.0 / SUM(COUNT(*)) OVER () AS proportion
    FROM uplift_train_raw
    GROUP BY treatment_flg, target
    ORDER BY treatment_flg, target
    """
).df()

display(treatment_target_profile)


,treatment_nulls,treatment_invalid,target_nulls,target_invalid
0,0,0,0,0


,treatment_flg,target,customers,proportion
0,0,0,39695,0.1984
1,0,1,60363,0.3018
2,1,0,36342,0.1817
3,1,1,63639,0.3181


In [240]:
uplift_relationship_check = duckdb_connection.execute(
    """ 
    SELECT
        COUNT(*) FILTER(
            WHERE c.client_id IS NULL
        ) AS uplift_train_clients_missing_from_clients
    FROM uplift_train_raw AS u
    LEFT JOIN clients_raw AS c
        USING (client_id)
    """
).df()

purchase_relationship_check = duckdb_connection.execute(
    """ 
    SELECT
        COUNT(*) FILTER(
            WHERE c.client_id IS NULL    
        ) AS purchases_row_with_unknow_client,
        COUNT(*) FILTER(
            WHERE pr.product_id IS NULL
        ) AS purchases_rows_with_unknown_product
    FROM purchases_raw as p
    LEFT JOIN clients_raw as c
        USING (client_id)
    LEFT JOIN products_raw AS pr
        USING (product_id)
    """
).df()

display(uplift_relationship_check)
display(purchase_relationship_check)

if uplift_relationship_check.iloc[0, 0] > 0:
    validation_issues.append("uplift_train contains unknown client_id values")

if purchase_relationship_check.iloc[0].sum() > 0:
    validation_issues.append("purchases contains unknown client_id or product_id values")

,uplift_train_clients_missing_from_clients
0,0


,purchases_row_with_unknow_client,purchases_rows_with_unknown_product
0,0,0


In [241]:
purchase_history_profile = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) AS purchase_rows,
        COUNT(DISTINCT client_id) AS purchasing_clients,
        COUNT(DISTINCT transaction_id) AS transactions,
        COUNT(DISTINCT product_id) AS purchased_products
    FROM purchases_raw
    """
).df()

display(purchase_history_profile)

,purchase_rows,purchasing_clients,transactions,purchased_products
0,45786568,400162,8045201,42530


In [242]:
NUMERIC_COLUMNS = {
    "clients": ["age"],
    "products": ["netto"],
    "purchases": [
        "regular_points_received",
        "express_points_received",
        "regular_points_spent",
        "express_points_spent",
        "purchase_sum",
        "product_quantity",
        "trn_sum_from_iss",
        "trn_sum_from_red",
    ],
}

CATEGORICAL_COLUMNS = {
    "clients": ["gender"],
    "products": [
        "level_1",
        "level_2",
        "level_3",
        "level_4",
        "segment_id",
        "brand_id",
        "vendor_id",
    ],
    "purchases": ["store_id"],
}

DATETIME_COLUMNS = {
    "clients": ["first_issue_date", "first_redeem_date"],
    "purchases": ["transaction_datetime"],
}

REQUIRED_NON_NULL = {
    "clients": {"client_id"},
    "products": {"product_id"},
    "purchases": {"client_id", "transaction_id", "transaction_datetime", "product_id"},
    "uplift_train": {"client_id", "treatment_flg", "target"},
}

In [243]:
NUMERIC_AGGREGATES = {
    "count": "COUNT({column})",
    "mean": "AVG({column})",
    "std": "STDDEV_SAMP({column})",
    "min": "MIN({column})",
    "25%": "APPROX_QUANTILE({column}, 0.25)",
    "50%": "APPROX_QUANTILE({column}, 0.50)",
    "75%": "APPROX_QUANTILE({column}, 0.75)",
    "max": "MAX({column})",
}


def numeric_profile(table_name, columns):
    expressions = []

    for column in columns:
        quoted = f'"{column}"'
        for metric, expression in NUMERIC_AGGREGATES.items():
            expressions.append(
                f'{expression.format(column=quoted)} AS "{column}__{metric}"'
            )

    result = duckdb_connection.execute(
        f"""
        SELECT {", ".join(expressions)}
        FROM {RELATIONS[table_name]}
        """
    ).df().iloc[0]

    return pd.DataFrame([
        {
            "table": table_name,
            "column": column,
            **{
                metric: result[f"{column}__{metric}"]
                for metric in NUMERIC_AGGREGATES
            },
        }
        for column in columns
    ])


numeric_profiles = pd.concat(
    [
        numeric_profile(table_name, columns)
        for table_name, columns in NUMERIC_COLUMNS.items()
    ],
    ignore_index=True,
)

display(numeric_profiles)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,table,column,count,mean,std,min,25%,50%,75%,max
0,clients,age,"400,162.0000",46.4881,43.8712,"-7,491.0000",33.0000,45.0000,59.0000,"1,901.0000"
1,products,netto,"43,035.0000",0.5370,8.2744,0.0000,0.1499,0.3053,0.5000,"1,150.0000"
2,purchases,regular_points_received,"45,786,568.0000",8.0499,12.6850,0.0000,1.3757,3.7678,10.3401,"2,399.0000"
3,purchases,express_points_received,"45,786,568.0000",0.0608,2.4262,0.0000,0.0000,0.0000,0.0000,300.0000
4,purchases,regular_points_spent,"45,786,568.0000",-5.3126,36.0365,"-5,066.0000",0.0000,0.0000,0.0000,0.0000
5,purchases,express_points_spent,"45,786,568.0000",-0.3181,3.2880,-300.0000,0.0000,0.0000,0.0000,0.0000
6,purchases,purchase_sum,"45,786,568.0000",777.5215,796.5350,0.0000,286.1743,539.5377,974.9256,"35,149.0400"
7,purchases,product_quantity,"45,786,568.0000",1.2472,3.1376,0.0000,1.0000,1.0000,1.0000,"14,941.0000"
8,purchases,trn_sum_from_iss,"45,786,568.0000",73.4884,87.5398,0.0000,30.0299,50.9685,89.9465,"35,149.0000"
9,purchases,trn_sum_from_red,"3,043,356.0000",76.7741,84.2711,0.0000,31.2307,54.5505,94.2973,"8,789.0000"


In [244]:
missing_profiles = []

for table_name, relation_name in RELATIONS.items():
    columns = [
        row[0]
        for row in duckdb_connection.execute(
            f"DESCRIBE {relation_name}"
        ).fetchall()
    ]

    expressions = [
        f'COUNT(*) FILTER (WHERE "{column}" IS NULL) AS "{column}"'
        for column in columns
    ]

    result = duckdb_connection.execute(
        f"""
        SELECT
            COUNT(*) AS total_rows,
            {", ".join(expressions)}
        FROM {relation_name}
        """
    ).df().iloc[0]

    total_rows = int(result["total_rows"])

    for column in columns:
        null_count = int(result[column])
        required = column in REQUIRED_NON_NULL.get(table_name, set())

        missing_profiles.append({
            "table": table_name,
            "column": column,
            "null_count": null_count,
            "null_rate": null_count / total_rows if total_rows else np.nan,
            "required_non_null": required,
        })

        if required and null_count:
            validation_issues.append(
                f"{table_name}.{column}: {null_count} unexpected null values"
            )

missing_profile = pd.DataFrame(missing_profiles)

display(
    missing_profile.sort_values(
        ["null_rate", "table", "column"],
        ascending=[False, True, True],
    )
)

,table,column,null_count,null_rate,required_non_null
28,purchases,trn_sum_from_red,42743212,0.9335,False
11,products,brand_id,5200,0.1208,False
2,clients,first_redeem_date,35469,0.0886,False
10,products,segment_id,1572,0.0365,False
12,products,vendor_id,34,0.0008,False
6,products,level_1,3,0.0001,False
7,products,level_2,3,0.0001,False
8,products,level_3,3,0.0001,False
9,products,level_4,3,0.0001,False
13,products,netto,3,0.0001,False


In [245]:
MAX_FULL_CATEGORIES = 20
TOP_CATEGORIES = 10

categorical_profiles = []
categorical_counts = {}

for table_name, columns in CATEGORICAL_COLUMNS.items():
    relation_name = RELATIONS[table_name]

    for column in columns:
        n_unique = duckdb_connection.execute(
            f"""
            SELECT COUNT(DISTINCT "{column}")
            FROM {relation_name}
            WHERE "{column}" IS NOT NULL
            """
        ).fetchone()[0]

        limit = "" if n_unique <= MAX_FULL_CATEGORIES else f"LIMIT {TOP_CATEGORIES}"

        counts = duckdb_connection.execute(
            f"""
            SELECT
                "{column}" AS value,
                COUNT(*) AS count,
            FROM {relation_name}
            WHERE "{column}" IS NOT NULL
            GROUP BY "{column}"
            ORDER BY count DESC
            {limit}
            """
        ).df()

        categorical_profiles.append({
            "table": table_name,
            "column": column,
            "n_unique": n_unique,
        })

        categorical_counts[(table_name, column)] = counts

display(pd.DataFrame(categorical_profiles))

for (table_name, column), counts in categorical_counts.items():
    print(f"\n{table_name}.{column}")
    display(counts)

,table,column,n_unique
0,clients,gender,3
1,products,level_1,3
2,products,level_2,42
3,products,level_3,201
4,products,level_4,790
5,products,segment_id,116
6,products,brand_id,4296
7,products,vendor_id,3193
8,purchases,store_id,13882



clients.gender


,value,count
0,U,185706
1,F,147649
2,M,66807



products.level_1


,value,count
0,e344ab2e71,22183
1,c3d3a8e8c6,16573
2,ec62ce61e3,4279


...

,value,count
0,0d6f137fb6,4344
1,4da2dc345f,3071
2,b06ace74de,385
3,ab230258e9,268
4,63ba6b7a61,136
5,7a282015f3,123
6,a548b9f2b8,116
7,74251e93eb,111
8,8188d00160,108
9,aa73f98d68,106



products.vendor_id


,value,count
0,43acd80c1a,1514
1,63243765ed,349
2,c4e167b91e,331
3,83f98e6dc3,328
4,4f276b13c1,323
5,e6af81215a,300
6,addfbe3485,224
7,41aa9501e1,223
8,3034fb4c4a,211
9,ef3b92f068,194



purchases.store_id


,value,count
0,cfbbd53ab7,18984
1,1f41964607,17436
2,a87bebd240,16833
3,f7390207ef,15856
4,3159cd57ba,15669
5,e3bf88fabf,15528
6,fac91d76e3,15219
7,c37dad9a51,15172
8,b61c786803,15070
9,dfb94aca9d,15051


In [246]:
product_binary_check = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE is_own_trademark IS NOT NULL
              AND is_own_trademark NOT IN (0, 1)
        ) AS invalid_is_own_trademark,
        COUNT(*) FILTER (
            WHERE is_alcohol IS NOT NULL
              AND is_alcohol NOT IN (0, 1)
        ) AS invalid_is_alcohol
    FROM products_raw
    """
).df()

display(product_binary_check)

if product_binary_check.iloc[0].sum() > 0:
    validation_issues.append("products contains invalid binary values")

,invalid_is_own_trademark,invalid_is_alcohol
0,0,0


In [247]:
range_checks = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE age IS NOT NULL AND age < 0
        ) AS negative_age
    FROM clients_raw
    """
).df()

product_range_checks = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE netto IS NOT NULL AND netto < 0
        ) AS negative_netto
    FROM products_raw
    """
).df()

display(range_checks)
display(product_range_checks)

if range_checks.iloc[0, 0] > 0:
    validation_issues.append("clients.age contains negative values")

if product_range_checks.iloc[0, 0] > 0:
    validation_issues.append("products.netto contains negative values")

,negative_age
0,96


,negative_netto
0,0


In [248]:
datetime_profiles = []

for table_name, columns in DATETIME_COLUMNS.items():
    relation_name = RELATIONS[table_name]

    for column in columns:
        min_date, max_date, future_count = duckdb_connection.execute(
            f"""
            SELECT
                MIN("{column}"),
                MAX("{column}"),
                COUNT(*) FILTER (
                    WHERE "{column}" > CURRENT_TIMESTAMP
                )
            FROM {relation_name}
            """
        ).fetchone()

        datetime_profiles.append({
            "table": table_name,
            "column": column,
            "min": min_date,
            "max": max_date,
            "future_count": future_count,
        })

        if future_count:
            validation_issues.append(
                f"{table_name}.{column}: {future_count} future timestamps"
            )

display(pd.DataFrame(datetime_profiles))

,table,column,min,max,future_count
0,clients,first_issue_date,2017-04-04 18:24:18,2019-03-15 21:50:56,0
1,clients,first_redeem_date,2017-04-11 09:42:20,2019-11-20 01:14:10,0
2,purchases,transaction_datetime,2018-11-21 21:02:33,2019-03-18 23:40:03,0


In [249]:
client_date_check = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE first_issue_date IS NOT NULL
              AND first_redeem_date IS NOT NULL
              AND first_redeem_date < first_issue_date
        ) AS redeem_before_issue
    FROM clients_raw
    """
).df()

display(client_date_check)

if client_date_check.iloc[0, 0] > 0:
    validation_issues.append(
        "clients contains first_redeem_date earlier than first_issue_date"
    )

,redeem_before_issue
0,536


# Structural Data Quality Checks

* Các bảng `clients`, `products` và `uplift_train` đều không có duplicate key. Cụ thể, `client_id` trong `clients` có 400,162 giá trị distinct trên 400,162 dòng; `product_id` trong `products` có 43,038 giá trị distinct trên 43,038 dòng; và `client_id` trong `uplift_train` có 200,039 giá trị distinct trên 200,039 dòng. Đồng thời, cả ba bảng đều không có null key. Điều này cho thấy các khóa định danh đang đảm bảo tính duy nhất và có thể được sử dụng để liên kết các bảng.

* Kiểm tra tính hợp lệ của hai biến quan trọng trong `uplift_train`: `treatment_flg` và `target`. Cả hai biến đều không có giá trị thiếu và không có giá trị ngoài miền `{0,1}`. Phân bố kết hợp giữa treatment và target cho thấy:

  * `treatment_flg = 0, target = 0`: 39,695 khách hàng (19.84%)
  * `treatment_flg = 0, target = 1`: 60,363 khách hàng (30.18%)
  * `treatment_flg = 1, target = 0`: 36,342 khách hàng (18.17%)
  * `treatment_flg = 1, target = 1`: 63,639 khách hàng (31.81%)

* Như vậy, cả treatment và control đều có số lượng khách hàng lớn và đều chứa cả outcome `0` và `1`. Điều này đảm bảo có đủ biến thiên trong treatment và outcome để tiếp tục phân tích sự khác biệt về outcome giữa các nhóm.

* Kiểm tra referential integrity cho thấy tất cả `client_id` trong `uplift_train` đều tồn tại trong bảng `clients`. Tương tự, toàn bộ các dòng trong `purchases` đều tham chiếu đến `client_id` hợp lệ trong `clients` và `product_id` hợp lệ trong `products`. Không phát hiện orphan records, cho thấy các quan hệ giữa các bảng đang nhất quán.

* Bảng `purchases` chứa 45,786,568 dòng giao dịch, 400,162 khách hàng có lịch sử mua hàng, 8,045,201 transaction và 42,530 sản phẩm xuất hiện trong lịch sử mua hàng. Đây là nguồn lịch sử giao dịch lớn, phù hợp để xây dựng các đặc trưng phản ánh hành vi mua hàng ở cấp khách hàng.

* Kiểm tra numeric columns cho thấy phần lớn các biến có phân bố và khoảng giá trị cần được xem xét ở bước EDA/feature engineering thay vì loại bỏ ngay. Đáng chú ý, `products.netto` không có giá trị âm, trong khi `clients.age` xuất hiện `96 giá trị âm`. Đây là một bất thường rõ ràng về domain và cần được xác định nguyên nhân trước khi sử dụng `age` làm feature.

* Missing-value check cho thấy các required fields như `client_id`, `transaction_id`, `transaction_datetime`, `product_id`, `treatment_flg` và `target` đều không có giá trị null. Tuy nhiên, một số thuộc tính không bắt buộc thì lại có missing đáng kể, đặc biệt `purchases.trn_sum_from_red` có 42,743,212 giá trị null (93.35%), `products.brand_id`, `clients.first_redeem_date` và `products.segment_id`. Một số cột khác như `vendor_id`, `level_1`–`level_4` và `netto` chỉ có số lượng missing rất nhỏ. Vì vậy, missingness không ảnh hưởng đến structural keys nhưng cần được phân tích trước khi feature engineering.

* Kiểm tra các binary flag của sản phẩm cho thấy `is_own_trademark` và `is_alcohol` không phát hiện giá trị ngoài miền.

* Kiểm tra datetime cho thấy không có timestamp nào nằm trong tương lai. `transaction_datetime` nằm trong khoảng từ `2018-11-21` đến `2019-03-18`; `first_issue_date` từ `2017-04-04` đến `2019-03-15`; và `first_redeem_date` từ `2017-04-11` đến `2019-11-20`.

* Tuy nhiên, kiểm tra quan hệ giữa các mốc thời gian phát hiện 536 trường hợp `first_redeem_date < first_issue_date`. Đây là một domain inconsistency cần được kiểm tra và xử lý trước khi sử dụng trực tiếp các biến ngày này để xây dựng đặc trưng liên quan đến vòng đời khách hàng.

Nhìn chung, cấu trúc dữ liệu và các quan hệ giữa bảng đang nhất quán: khóa định danh không bị trùng hoặc thiếu, `treatment` và `target` có miền giá trị hợp lệ, không có orphan records, các binary flag hợp lệ và không có timestamp trong tương lai. Tuy nhiên, dữ liệu **chưa hoàn toàn sạch ở cấp value quality**: tồn tại missing values đáng kể ở một số thuộc tính, 96 giá trị `age` âm và 536 trường hợp bất thường trong quan hệ giữa `first_redeem_date` và `first_issue_date`. Các vấn đề này cần được ghi nhận và xử lý ở bước Data Understanding / Feature Engineering thay vì xem toàn bộ dữ liệu là hoàn toàn hợp lệ ngay từ đầu.


## Investigation of Data Quality Issues


In [250]:
age_anomalies = duckdb_connection.execute(
    """
    SELECT
        age,
        COUNT(*) AS clients,
        ROUND(
            COUNT(*) * 100.0 /
            SUM(COUNT(*)) OVER (),
            2
        ) AS pct_of_anomalies
    FROM clients_raw
    WHERE age < 0 OR age > 100
    GROUP BY age
    ORDER BY clients DESC
    """
).df()

display(age_anomalies)

,age,clients,pct_of_anomalies
0,115,405,35.3700
1,119,397,34.6700
2,944,7,0.6100
3,949,7,0.6100
4,952,6,0.5200
...,...,...,...
185,-2,1,0.0900
186,1011,1,0.0900
187,-2969,1,0.0900
188,930,1,0.0900


In [251]:
age_anomaly_rows = duckdb_connection.execute(
    """
    SELECT
        c.client_id,
        c.age,
        c.gender,
        c.first_issue_date,
        c.first_redeem_date,
        u.treatment_flg,
        u.target
    FROM clients_raw AS c
    LEFT JOIN uplift_train_raw AS u USING (client_id)
    WHERE c.age < 0 OR c.age > 100
    ORDER BY c.age
    """
).df()

display(age_anomaly_rows)

,client_id,age,gender,first_issue_date,first_redeem_date,treatment_flg,target
0,6c43d328e8,-7491,U,2017-07-24 12:30:49,2019-02-27 14:58:25,0,1
1,3b030e7fa6,-7158,U,2018-11-16 18:50:20,2018-12-07 19:40:27,0,0
2,86d1dc8761,-5973,U,2018-10-15 22:45:03,2019-01-26 16:24:53,1,1
3,8a0e463a19,-5953,U,2017-05-04 16:36:23,2017-06-13 16:57:04,0,0
4,71c2137d2d,-5949,U,2018-11-25 13:28:17,2019-03-22 10:17:38,1,1
...,...,...,...,...,...,...,...
1140,1350cceaf5,1717,U,2018-11-17 21:49:47,NaT,0,0
1141,768f4e55ac,1800,U,2018-10-31 18:26:43,2019-07-19 17:28:05,<NA>,<NA>
1142,5252823991,1841,U,2018-10-30 08:59:55,2019-02-22 08:41:34,1,0
1143,6edee96676,1852,U,2017-11-16 16:32:46,2018-10-18 11:29:48,1,1


In [252]:
duckdb_connection.execute(
    """
    CREATE OR REPLACE TEMP VIEW age_anomaly_clients AS
    SELECT *
    FROM clients_raw
    WHERE age < 0 OR age > 100
    """
)

In [253]:
age_anomaly_context = duckdb_connection.execute(
    """
    SELECT
        age,
        COUNT(*) AS clients,
        COUNT(*) FILTER (WHERE gender = 'U') AS unknown_gender,
        ROUND(COUNT(*) FILTER (WHERE gender = 'U') * 100.0 / COUNT(*), 2) AS pct_unknown_gender,
        COUNT(*) FILTER (WHERE first_redeem_date IS NULL) AS missing_redeem_date
    FROM age_anomaly_clients
    GROUP BY age
    ORDER BY clients DESC
    LIMIT 20
    """
).df()

display(age_anomaly_context)

,age,clients,unknown_gender,pct_unknown_gender,missing_redeem_date
0,115,405,2,0.4900,114
1,119,397,363,91.4400,40
2,949,7,7,100.0000,1
3,944,7,7,100.0000,1
4,934,6,6,100.0000,0
5,946,6,6,100.0000,0
6,952,6,6,100.0000,0
7,959,5,5,100.0000,1
8,109,5,1,20.0000,1
9,118,5,1,20.0000,1


In [254]:
age_gender_pattern = duckdb_connection.execute(
    """
    SELECT
        age,
        gender,
        COUNT(*) AS clients
    FROM clients_raw
    WHERE age < 0 OR age > 100
    GROUP BY age, gender
    ORDER BY age, clients DESC
    """
).df()

display(
    age_gender_pattern[
        age_gender_pattern["age"].isin([115, 119])
    ]
)

,age,gender,clients
79,115,M,402
80,115,U,2
81,115,F,1
87,119,U,363
88,119,F,19
89,119,M,15


In [255]:
special_age_time_pattern = duckdb_connection.execute(
    """
    SELECT
        age,
        DATE_TRUNC('month', first_issue_date) AS issue_month,
        COUNT(*) AS clients
    FROM clients_raw
    WHERE age IN (115, 119)
    GROUP BY age, issue_month
    ORDER BY age, issue_month
    """
).df()

display(special_age_time_pattern)

,age,issue_month,clients
0,115,2017-04-01,14
1,115,2017-05-01,18
2,115,2017-06-01,46
3,115,2017-07-01,51
4,115,2017-08-01,42
5,115,2017-09-01,28
6,115,2017-10-01,25
7,115,2017-11-01,35
8,115,2017-12-01,46
9,115,2018-01-01,12


In [256]:
duckdb_connection.execute(
    """
    CREATE OR REPLACE TEMP VIEW client_purchase_profile AS
    SELECT
        client_id,
        COUNT(*) AS purchase_rows,
        COUNT(DISTINCT transaction_id) AS transactions,
        SUM(purchase_sum) AS total_purchase_sum,
        AVG(purchase_sum) AS avg_purchase_sum,
        AVG(product_quantity) AS avg_product_quantity,
        SUM(regular_points_received) AS regular_points_received,
        SUM(regular_points_spent) AS regular_points_spent,
        MIN(transaction_datetime) AS first_purchase,
        MAX(transaction_datetime) AS last_purchase
    FROM purchases_raw
    GROUP BY client_id
    """
)

In [257]:
age_group_purchase_profile = duckdb_connection.execute(
    """
    SELECT
        CASE
            WHEN c.age < 0 OR c.age > 100 THEN 'age_anomaly'
            ELSE 'normal_age'
        END AS age_group,
        COUNT(*) AS clients,
        AVG(p.transactions) AS avg_transactions,
        APPROX_QUANTILE(p.transactions, 0.5) AS median_transactions,
        AVG(p.purchase_rows) AS avg_purchase_rows,
        APPROX_QUANTILE(p.purchase_rows, 0.5) AS median_purchase_rows,
        AVG(p.total_purchase_sum) AS avg_total_purchase_sum,
        APPROX_QUANTILE(p.total_purchase_sum, 0.5) AS median_total_purchase_sum
    FROM clients_raw AS c
    LEFT JOIN client_purchase_profile AS p USING (client_id)
    GROUP BY age_group
    """
).df()

display(age_group_purchase_profile)

,age_group,clients,avg_transactions,median_transactions,avg_purchase_rows,median_purchase_rows,avg_total_purchase_sum,median_total_purchase_sum
0,age_anomaly,1145,17.8777,13,104.1441,78,"83,592.7442","40,932.4393"
1,normal_age,399017,20.1113,15,114.4496,85,"88,979.4828","44,016.7731"


In [258]:
age_value_purchase_profile = duckdb_connection.execute(
    """
    SELECT
        a.age,
        COUNT(*) AS clients,
        AVG(p.transactions) AS avg_transactions,
        APPROX_QUANTILE(p.transactions, 0.5) AS median_transactions,
        AVG(p.total_purchase_sum) AS avg_total_purchase_sum,
        APPROX_QUANTILE(p.total_purchase_sum, 0.5) AS median_total_purchase_sum
    FROM age_anomaly_clients AS a
    LEFT JOIN client_purchase_profile AS p USING (client_id)
    GROUP BY a.age
    ORDER BY clients DESC
    LIMIT 20
    """
).df()

display(age_value_purchase_profile)

,age,clients,avg_transactions,median_transactions,avg_total_purchase_sum,median_total_purchase_sum
0,115,405,14.7975,11,"70,220.7868","34,336.5822"
1,119,397,21.1209,17,"99,237.7378","48,004.2175"
2,944,7,23.0000,14,"65,961.5143","25,145.4000"
3,949,7,25.2857,22,"106,311.1014","45,752.9900"
4,952,6,15.3333,10,"45,323.8033","44,115.5000"
5,946,6,43.5000,36,"101,512.8500","101,521.2450"
6,934,6,23.3333,20,"170,997.6033","33,171.8200"
7,956,5,13.0000,10,"54,254.6760","30,410.0000"
8,-943,5,18.0000,22,"56,616.6580","56,443.7700"
9,109,5,22.2000,21,"72,198.1420","56,511.5200"


In [259]:
age_anomaly_uplift_profile = duckdb_connection.execute(
    """
    SELECT
        u.treatment_flg,
        u.target,
        COUNT(*) AS clients
    FROM age_anomaly_clients AS a
    JOIN uplift_train_raw AS u USING (client_id)
    GROUP BY u.treatment_flg, u.target
    ORDER BY u.treatment_flg, u.target
    """
).df()

display(age_anomaly_uplift_profile)

,treatment_flg,target,clients
0,0,0,135
1,0,1,151
2,1,0,128
3,1,1,163


Kiểm tra `age` phát hiện **1,145 khách hàng** có giá trị bất hợp lý theo rule `age < 0` hoặc `age > 100`, với **190 giá trị bất thường khác nhau**. Khoảng giá trị trải từ `-7491` đến `1901`, cho thấy đây không phải các tuổi thực tế hợp lệ.

Hai giá trị xuất hiện nhiều nhất là:

- `age = 115`: 405 khách hàng, chiếm 35.37% số age anomaly.
- `age = 119`: 397 khách hàng, chiếm 34.67% số age anomaly.

Hai giá trị này chiếm khoảng 70% tổng số age anomaly, cho thấy vấn đề có tính hệ thống chứ không chỉ là một vài outlier ngẫu nhiên.

Kiểm tra thêm demographic context cho thấy hai nhóm cũng có pattern khác biệt rõ ràng:

- `age = 115`: 402/405 khách hàng có `gender = M`.
- `age = 119`: 363/397 khách hàng có `gender = U`.

Các giá trị `115` và `119` xuất hiện xuyên suốt giai đoạn từ 2017 đến 2019, không tập trung vào một batch hoặc một thời điểm cụ thể. Ngoài ra còn tồn tại nhiều giá trị hoàn toàn phi thực tế như `949`, `-943`, `-7491`, `1717`, `1901`, cho thấy cột `age` có các giá trị bị lỗi hoặc được encode không đúng.

Tuy nhiên, khi đối chiếu với lịch sử mua hàng, nhóm khách hàng có age bất thường vẫn có hành vi giao dịch tương đối giống với nhóm có age hợp lệ:

- Mdian transactions: 13 so với 15
- Median purchase rows: 78 so với 86
- Median total purchase sum: khoảng 40.9k so với 44.0k.

Treatment và target của các khách hàng này cũng vẫn có đủ cả bốn nhóm `(T, Y)`, không cho thấy các record này bị lỗi toàn bộ.

In [260]:
quantity_tail = duckdb_connection.execute(
    """
    SELECT
        APPROX_QUANTILE(product_quantity, 0.99) AS q99,
        APPROX_QUANTILE(product_quantity, 0.999) AS q999,
        APPROX_QUANTILE(product_quantity, 0.9999) AS q9999,
        MAX(product_quantity) AS max
    FROM purchases_raw
    """
).df()

display(quantity_tail)

,q99,q999,q9999,max
0,4.9490,12.6042,40.6503,"14,941.0000"


In [261]:
quantity_cutoff = quantity_tail.loc[0, "q9999"]

extreme_product_profile = duckdb_connection.execute(
    f"""
    WITH extreme_products AS (
        SELECT DISTINCT product_id
        FROM purchases_raw
        WHERE product_quantity > {quantity_cutoff}
    )
    SELECT
        p.product_id,
        COUNT(*) AS purchase_rows,
        COUNT(DISTINCT p.client_id) AS clients,
        MIN(p.product_quantity) AS min_quantity,
        APPROX_QUANTILE(p.product_quantity, 0.50) AS median_quantity,
        APPROX_QUANTILE(p.product_quantity, 0.90) AS q90_quantity,
        APPROX_QUANTILE(p.product_quantity, 0.99) AS q99_quantity,
        MAX(p.product_quantity) AS max_quantity,
        COUNT(*) FILTER (
            WHERE p.product_quantity > {quantity_cutoff}
        ) AS extreme_rows
    FROM purchases_raw AS p
    JOIN extreme_products AS e USING (product_id)
    GROUP BY p.product_id
    ORDER BY max_quantity DESC
    """
).df()

display(extreme_product_profile.head(30))

,product_id,purchase_rows,clients,min_quantity,median_quantity,q90_quantity,q99_quantity,max_quantity,extreme_rows
0,ce5dddfb68,10733,7985,1.0000,1.0000,1.9054,2.9267,"14,941.0000",1
1,e7ad2b87e1,20609,14375,1.0000,2.0000,4.0000,15.5120,"9,228.0000",18
2,5481e41adf,7686,6912,1.0000,1.0000,2.0000,4.0000,"9,227.0000",1
3,47561f90e6,2210,1520,1.0000,1.0000,2.0000,3.6806,660.0000,1
4,ee4f7132c3,27376,13385,1.0000,2.0000,6.0000,16.5121,648.0000,38
5,4009f09b04,1824586,275599,1.0000,1.0000,1.0012,3.0000,620.0000,53
6,4dcf79043e,370451,149865,1.0000,1.0000,3.0000,11.9526,500.0000,388
7,ea27d5dc75,147477,71926,0.0000,2.0000,3.0000,8.1209,300.0000,163
8,a755a6f2e6,13811,10695,1.0000,1.0000,2.0000,4.0532,278.0000,2
9,197c432c53,12912,7963,1.0000,2.0000,5.0000,10.1694,254.0000,3


In [262]:
quantity_extreme_summary = duckdb_connection.execute(
    f"""
    SELECT
        COUNT(*) AS extreme_rows,
        COUNT(DISTINCT product_id) AS products,
        COUNT(DISTINCT client_id) AS clients,
        COUNT(DISTINCT transaction_id) AS transactions
    FROM purchases_raw
    WHERE product_quantity > {quantity_cutoff}
    """
).df()

display(quantity_extreme_summary)

,extreme_rows,products,clients,transactions
0,1735,348,942,1656


In [263]:
netto_extremes = duckdb_connection.execute(
    """
    SELECT
        product_id,
        netto,
        level_1,
        level_2,
        level_3,
        level_4,
        segment_id,
        brand_id
    FROM products_raw
    ORDER BY netto DESC NULLS LAST
    LIMIT 30
    """
).df()

display(netto_extremes)

,product_id,netto,level_1,level_2,level_3,level_4,segment_id,brand_id
0,ec9077783d,"1,150.0000",c3d3a8e8c6,f93982269d,0bcfc6519b,b4b0e4c470,259.0000,None
1,98528c8b99,600.0000,c3d3a8e8c6,f93982269d,0bcfc6519b,b4b0e4c470,259.0000,None
2,2fbd8c3d9c,600.0000,c3d3a8e8c6,f93982269d,0bcfc6519b,b4b0e4c470,259.0000,None
3,61a26b3d4b,430.0000,e344ab2e71,52f13dac0c,6b55683dad,56426fcb60,105.0000,4da2dc345f
4,0020caa486,400.0000,e344ab2e71,52f13dac0c,6b55683dad,56426fcb60,105.0000,4da2dc345f
5,b46720cd94,350.0000,c3d3a8e8c6,f93982269d,0bcfc6519b,fc32a80dcd,259.0000,85606f387b
6,44f2e529ac,316.8000,c3d3a8e8c6,ad2b2e17d2,ca69ed9de2,8bbeabc581,212.0000,563ceeb1d9
7,42511cfd14,316.8000,c3d3a8e8c6,ad2b2e17d2,ca69ed9de2,8bbeabc581,212.0000,563ceeb1d9
8,50e2016e67,260.0000,e344ab2e71,52f13dac0c,6b55683dad,56426fcb60,105.0000,4da2dc345f
9,d07c11c602,259.2000,c3d3a8e8c6,ad2b2e17d2,ca69ed9de2,8bbeabc581,212.0000,563ceeb1d9


In [264]:
extreme_quantity_netto = duckdb_connection.execute(
    f"""
    SELECT
        CASE
            WHEN pr.netto < 0.05 THEN '<0.05'
            WHEN pr.netto < 0.10 THEN '0.05-0.10'
            WHEN pr.netto < 0.50 THEN '0.10-0.50'
            WHEN pr.netto <= 1.00 THEN '0.50-1.00'
            ELSE '>1.00'
        END AS netto_group,
        COUNT(*) AS extreme_rows,
        COUNT(DISTINCT p.product_id) AS products,
        AVG(p.product_quantity) AS avg_quantity,
        MAX(p.product_quantity) AS max_quantity
    FROM purchases_raw AS p
    JOIN products_raw AS pr USING (product_id)
    WHERE p.product_quantity > {quantity_cutoff}
      AND pr.netto IS NOT NULL
    GROUP BY netto_group
    ORDER BY netto_group
    """
).df()

display(extreme_quantity_netto)

,netto_group,extreme_rows,products,avg_quantity,max_quantity
0,0.05-0.10,330,111,58.8879,278.0000
1,0.10-0.50,283,118,146.3993,"14,941.0000"
2,0.50-1.00,801,52,71.9288,500.0000
3,<0.05,311,62,101.4148,"9,228.0000"
4,>1.00,10,5,66.7000,124.0000


In [265]:
top_extreme_products = duckdb_connection.execute(
    """
    SELECT DISTINCT product_id
    FROM purchases_raw
    ORDER BY product_quantity DESC
    LIMIT 3
    """
).df()["product_id"].tolist()

top_extreme_history = duckdb_connection.execute(
    f"""
    SELECT
        product_id,
        transaction_id,
        client_id,
        transaction_datetime,
        product_quantity,
        trn_sum_from_iss,
        purchase_sum
    FROM purchases_raw
    WHERE product_id IN ({",".join(repr(x) for x in top_extreme_products)})
    ORDER BY product_id, product_quantity DESC
    """
).df()

display(top_extreme_history.head(50))

,product_id,transaction_id,client_id,transaction_datetime,product_quantity,trn_sum_from_iss,purchase_sum
0,37f952f663,caa59e52e5,d70be3b2cc,2018-12-22 15:47:11,27.0000,378.0000,"6,535.6300"
1,37f952f663,5b870c83c9,0fa9e4771b,2019-01-29 15:37:32,27.0000,378.0000,"2,109.0000"
2,37f952f663,ef28cbb98d,4b16c66f5a,2019-02-24 14:51:28,15.0000,209.0000,559.0000
3,37f952f663,4b3a2164c9,7b4a0e1d31,2018-12-27 14:14:26,14.0000,210.0000,873.0000
4,37f952f663,7390f6e6a4,a3388b926d,2018-11-23 13:55:15,13.0000,182.0000,438.0000
5,37f952f663,0538d7ba4a,1027763075,2019-02-21 06:50:54,10.0000,146.0000,"1,125.2600"
6,37f952f663,591fff5941,72e992f064,2019-03-13 09:47:34,10.0000,131.0000,"1,729.3500"
7,37f952f663,7c121b5f9c,515385a884,2019-03-04 19:14:55,9.0000,130.0000,"2,480.0000"
8,37f952f663,4b780a2a8f,3cbd914698,2018-12-28 14:35:44,6.0000,86.0000,"1,052.9200"
9,37f952f663,bc48c01198,53c73fb9d3,2019-01-27 10:55:41,6.0000,79.0000,309.0000


Kiểm tra phân bố của `product_quantity` cho thấy biến này có **right-skew rất mạnh** và tồn tại một số giá trị cực lớn so với phần lớn dữ liệu.

Các mốc phân vị cho thấy:

- 99% purchase rows có `product_quantity <= 4.93`;
- 99.9% có `product_quantity <= 12.73`;
- 99.99% có `product_quantity <= 39.11`;
- trong khi giá trị lớn nhất lên tới `14,941`.

Khi kiểm tra lịch sử đầy đủ của các sản phẩm có quantity cực lớn, phát hiện hai pattern khác nhau.

Một số sản phẩm có phần lớn giao dịch với quantity rất nhỏ nhưng chỉ xuất hiện một vài giá trị cực đoan:

- `product_id = ce5dddfb68` có 10,733 purchase rows, median quantity = 1, q99 ≈ 2.96, nhưng chỉ có 1 dòng đạt `14,941`;
- `product_id = 5481e41adf` có 7,686 purchase rows, median quantity = 1, q99 = 4, nhưng chỉ có 1 dòng đạt `9,227`;
- `product_id = 47561f90e6` có 2,210 purchase rows, median quantity = 1, q99 ≈ 3.65, nhưng chỉ có 1 dòng đạt `660`.

Những trường hợp này giống isolated extreme observations vì giá trị lớn nhất cách rất xa hành vi thông thường của cùng một product.

Tuy nhiên, một số sản phẩm khác lại có high quantity xuất hiện lặp lại nhiều lần:

- `product_id = 4dcf79043e` có 370,451 purchase rows, q99 ≈ 11.83 và 388 dòng vượt ngưỡng q99.99 chung;
- `product_id = ea27d5dc75` có 147,477 purchase rows, q99 ≈ 8.22 và 163 extreme rows;
- `product_id = f2293d7dfa` có 16,852 purchase rows, q99 ≈ 28.16 và 123 extreme rows.

Điều này cho thấy high quantity không phải lúc nào cũng là lỗi đơn lẻ. Với một số sản phẩm, quantity lớn có thể phản ánh cách đo lường hoặc đặc tính mua hàng riêng của sản phẩm.

Tổng cộng có **1,735 purchase rows** vượt ngưỡng q99.99 (~39.11), trải trên:

- 348 sản phẩm;
- 942 khách hàng;
- 1,656 transaction.

Số lượng này rất nhỏ so với hơn 45 triệu purchase rows, nhưng các extreme values không tập trung vào duy nhất một product hoặc một loại transaction.

Kiểm tra thêm theo `netto` cũng không cho thấy một quan hệ đơn giản có thể dùng để xác định lỗi. Extreme quantity xuất hiện ở nhiều nhóm trọng lượng sản phẩm khác nhau, bao gồm cả sản phẩm có `netto` rất nhỏ và sản phẩm có `netto` khoảng 0.5–1.0. Vì vậy không có đủ bằng chứng để kết luận rằng quantity lớn là do một quy ước trọng lượng cụ thể.

Ngoài ra, các transaction có quantity lớn vẫn có `purchase_sum` và `trn_sum_from_iss` hợp lệ, cho thấy các dòng này không phải các record bị hỏng toàn bộ.


## Data Cleaning